# 06 - Quantificadores e Predicados em Redes de Sensores
Este notebook modela a lógica de primeira ordem aplicável à rede de sensores da Estação de Reabastecimento de Hidrogênio, utilizando quantificadores universais ($\forall$) e existenciais ($\exists$) para validar as condições operacionais dos Setores 100, 200 e 300.

In [1]:
# Mapeamento dos Sensores e Estados da Estacao SCADA-Core (ISA-5.1)
leituras_sensores = {
    # Setor 100: Armazenamento
    "PT-101": 380,   # bar (Limite: 400)
    "PT-102": 680,   # bar (Limite: 700)
    "PT-103": 950,   # bar (Limite: 1000)
    "PT-104": 355,   # bar (Set: >350)
    "PT-105": 655,   # bar (Set: >650)
    "PT-106": 960,   # bar (Set: >950)
    "TT-101": 42.0,  # degC (Limite: 85)
    "TT-102": 51.0,  # degC (Limite: 85)
    "TT-103": 48.0,  # degC (Limite: 85)
    "AT-101": 5.0,   # % LIE (Limite: 25)
    "AT-102": 0.0,   # % LIE (Limite: 25)
    "AT-103": 1.0,   # % LIE (Limite: 25)
    "ESD-100": False,# Botao de emergencia manual
    
    # Setor 200: Condicionamento
    "TT-201": -42.0, # degC (Limite <= -40)
    "PT-201": 700,   # bar
    "M-201": True,   # Contator Chiller Ligado
    "XV-201": True,  # Valvula de Entrada Chiller Aberta
    
    # Setor 300: Dispensacao
    "HS-301": True,  # Botao Inicio Operador
    "COM-301": True, # Comunicacao J2799 Veiculo
    "BV-301": True,  # Breakaway Integro
    "PT-301": 700,   # bar Enchimento
    "TT-301": 35.0,  # degC Veiculo (Limite: 85)
    "AT-301": 0.0    # % LIE Dispensador (Limite: 25)
}

# --- Predicados Lógicos ---
def P_crit(tag, valor):
    limites = {"PT-101": 400, "PT-102": 700, "PT-103": 1000}
    return valor > limites[tag] if tag in limites else False

def T_crit(tag, valor):
    if tag in ["TT-101", "TT-102", "TT-103", "TT-301"]:
        return valor > 85.0
    return False

def G_crit(tag, valor):
    if tag in ["AT-101", "AT-102", "AT-103", "AT-301"]:
        return valor > 25.0
    return False

def R_ok(valor_tt201):
    return valor_tt201 <= -40.0

# Quantificador Existencial: existe falha no Setor 100?
sensores_s100_criticos = [
    P_crit("PT-101", leituras_sensores["PT-101"]),
    P_crit("PT-102", leituras_sensores["PT-102"]),
    P_crit("PT-103", leituras_sensores["PT-103"]),
    T_crit("TT-101", leituras_sensores["TT-101"]),
    T_crit("TT-102", leituras_sensores["TT-102"]),
    T_crit("TT-103", leituras_sensores["TT-103"]),
    G_crit("AT-101", leituras_sensores["AT-101"]),
    G_crit("AT-102", leituras_sensores["AT-102"]),
    G_crit("AT-103", leituras_sensores["AT-103"])
]

falha_existencial_s100 = any(sensores_s100_criticos)
trip_sis = falha_existencial_s100 or leituras_sensores["ESD-100"]

# Quantificador Universal: todos os parametros seguros no Armazenamento?
armazenamento_integro = not falha_existencial_s100

# Avaliacao do Permissivo XV-301
permissivo_xv301 = all([
    armazenamento_integro,
    R_ok(leituras_sensores["TT-201"]),
    leituras_sensores["M-201"],
    leituras_sensores["COM-301"],
    leituras_sensores["BV-301"],
    leituras_sensores["HS-301"],
    not T_crit("TT-301", leituras_sensores["TT-301"]),
    not G_crit("AT-301", leituras_sensores["AT-301"]),
    not leituras_sensores["ESD-100"]
])

print("=== RELATÓRIO DE MONITORAMENTO DA REDE SCADA ===")
print(f"[SIS] Trip de Emergência Ativado: {trip_sis}")
print(f"[SETOR 100] Integridade Universal do Banco: {armazenamento_integro}")
print(f"[SETOR 200] Resfriamento (TT-201 <= -40°C): {R_ok(leituras_sensores['TT-201'])}")
print(f"[SETOR 300] Permissivo da Válvula XV-301: {permissivo_xv301}")

=== RELATÓRIO DE MONITORAMENTO DA REDE SCADA ===
[SIS] Trip de Emergência Ativado: False
[SETOR 100] Integridade Universal do Banco: True
[SETOR 200] Resfriamento (TT-201 <= -40°C): True
[SETOR 300] Permissivo da Válvula XV-301: True
